# Night LST on night-time lights: mining IV and the PM2.5 channel

**Outcome** `lst_night_mean`; **regressor** `log1p(ntl_harm)`; **instrument** `mine_count_20km`.
Pixel + country x year FE, SE clustered by country, full 10 km panel (`duckreg`, same setup as `regression.ipynb`).

| col | spec |
|---|---|
| (1) | TWFE OLS `lst_night ~ log_ntl` |
| (2) | 2SLS, `log_ntl` instrumented by mines 20 km (+ first stage) |
| (3) | reduced form `lst_night ~ mines` |
| (4) | TWFE OLS `lst_night ~ log_ntl + pm25` |
| (5) | reduced form `pm25 ~ mines` |

Cols (1)-(3) use pixel-years with night LST, NTL and mine counts; (4)-(5) additionally require `pm25`.
Note: PM2.5 is a mediator (`docs/data/acag/README.md`), so (4) is a mechanism check, not a better-identified estimate.

In [1]:
import sys, os, time
from pathlib import Path
import duckdb
import pandas as pd

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DUCKREG_PATH = os.environ.get("DUCKREG_PATH", str(PROJECT.parent / "duckreg"))
if DUCKREG_PATH not in sys.path:
    sys.path.insert(0, DUCKREG_PATH)
from duckreg import duckreg

# silence DuckDB's per-query progress bar in duckreg's own connections
import duckreg.estimators.base as _dre_base
_dre_orig_init = _dre_base.DuckEstimator._init_connection
def _dre_quiet_init(self):
    _dre_orig_init(self)
    try:
        self.conn.execute("SET enable_progress_bar = false")
    except Exception:
        pass
_dre_base.DuckEstimator._init_connection = _dre_quiet_init

GRID  = "10km"
SRC   = f"{PROJECT}/data/assembled/grid={GRID}/shake=base/ix=*/iy=*/*.parquet"
PANEL = f"{PROJECT}/data/_regression_panel_night_{GRID}.parquet"   # derived cache
OUT   = PROJECT / "output" / "tables"

Y, X, Z, PM = "lst_night_mean", "log_ntl", "mine_count_20km", "pm25"
FE    = "pixel_id + GID_0^year"
CL    = {"CRV1": "GID_0"}
PMSUB = f"{PM} IS NOT NULL"
DR    = dict(fitter="auto", compression=5, threads=8, memory_limit="24GB")

In [2]:
if not os.path.exists(PANEL):
    con = duckdb.connect()
    con.execute("PRAGMA threads=8")
    con.execute("SET enable_progress_bar = false")
    t = time.time()
    con.execute(f"""
        COPY (
          SELECT pixel_id, year, GID_0, {Y}, ntl_harm,
                 ln(ntl_harm + 1.0) AS {X}, {Z}, {PM}
          FROM read_parquet('{SRC}', union_by_name = true, hive_partitioning = true)
          WHERE {Y} IS NOT NULL AND ntl_harm IS NOT NULL AND {Z} IS NOT NULL
        ) TO '{PANEL}' (FORMAT parquet, COMPRESSION zstd)
    """)
    con.close()
    print(f"built {PANEL} in {time.time()-t:.0f}s")

def nobs(subset=None):
    w = f"WHERE {subset}" if subset else ""
    return duckdb.sql(
        f"SELECT count(*), count(DISTINCT pixel_id), count(DISTINCT GID_0), min(year), max(year) "
        f"FROM read_parquet('{PANEL}') {w}"
    ).fetchone()

S_MAIN, S_PM = nobs(), nobs(PMSUB)
print("main sample  (rows, pixels, countries, y0, y1):", S_MAIN)
print("PM2.5 sample (rows, pixels, countries, y0, y1):", S_PM)

built /Users/felixschulz/Library/CloudStorage/OneDrive-Personal/Dokumente/Job/UNI/Basel/Research/growth-and-temperature/data/_regression_panel_night_10km.parquet in 5s


main sample  (rows, pixels, countries, y0, y1): (25268139, 1205481, 253, 2002, 2022)
PM2.5 sample (rows, pixels, countries, y0, y1): (25268073, 1205471, 253, 2002, 2022)


In [3]:
def fit(fml, subset=None):
    t = time.time()
    m = duckreg(fml, data=PANEL, se_method=CL, subset=subset, **DR)
    print(f"{time.time()-t:5.0f}s  {fml}  [{subset or 'full'}]")
    return m

def get(m, term):
    """(estimate, se, p) for one term; the IV endog term is renamed by duckreg."""
    td = m.tidy().set_index("variable")
    if term not in td.index:
        term = next(i for i in td.index if term in i and i != "Intercept")
    r = td.loc[term]
    return float(r["estimate"]), float(r["std_error"]), float(r["p_value"])

M = {}
M["c1"]   = fit(f"{Y} ~ {X} | {FE}")
M["fs"]   = fit(f"{X} ~ {Z} | {FE}")
M["c2"]   = fit(f"{Y} ~ 1 | {FE} | ({X} ~ {Z})")
M["c3"]   = fit(f"{Y} ~ {Z} | {FE}")
M["c4"]   = fit(f"{Y} ~ {X} + {PM} | {FE}", subset=PMSUB)
M["c5"]   = fit(f"{PM} ~ {Z} | {FE}", subset=PMSUB)
# comparability checks on the PM2.5 sample
M["c1pm"] = fit(f"{Y} ~ {X} | {FE}", subset=PMSUB)
M["fspm"] = fit(f"{X} ~ {Z} | {FE}", subset=PMSUB)

  196s  lst_night_mean ~ log_ntl | pixel_id + GID_0^year  [full]


  184s  log_ntl ~ mine_count_20km | pixel_id + GID_0^year  [full]


  241s  lst_night_mean ~ 1 | pixel_id + GID_0^year | (log_ntl ~ mine_count_20km)  [full]


  180s  lst_night_mean ~ mine_count_20km | pixel_id + GID_0^year  [full]


  273s  lst_night_mean ~ log_ntl + pm25 | pixel_id + GID_0^year  [pm25 IS NOT NULL]


  179s  pm25 ~ mine_count_20km | pixel_id + GID_0^year  [pm25 IS NOT NULL]


  177s  lst_night_mean ~ log_ntl | pixel_id + GID_0^year  [pm25 IS NOT NULL]


  165s  log_ntl ~ mine_count_20km | pixel_id + GID_0^year  [pm25 IS NOT NULL]


## Table

In [4]:
def star(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .1 else ""

def cell(m, term, d=4):
    b, s, p = get(m, term)
    return f"{b:.{d}f}{star(p)}", f"({s:.{d}f})"

cols = ["(1) TWFE", "(2) 2SLS", "(3) RF: Temp", "(4) TWFE + PM2.5", "(5) RF: PM2.5"]
rows = {}
def put(label, entries):
    b = dict.fromkeys(cols, ""); s = dict.fromkeys(cols, "")
    for c, (m, term) in entries.items():
        b[c], s[c] = cell(M[m], term)
    rows[label] = b; rows[label + " se"] = s

put("log(1+NTL)",          {cols[0]: ("c1", X), cols[1]: ("c2", X), cols[3]: ("c4", X)})
put("Mines within 20 km",  {cols[2]: ("c3", Z), cols[4]: ("c5", Z)})
put("PM2.5",               {cols[3]: ("c4", PM)})
put("First stage: Mines 20 km -> log(1+NTL)", {cols[1]: ("fs", Z)})
fb, fse, _ = get(M["fs"], Z)
rows["First-stage F (cluster-robust)"] = {**dict.fromkeys(cols, ""), cols[1]: f"{(fb / fse) ** 2:.1f}"}

samp = lambda i: S_PM if i >= 3 else S_MAIN
rows["Dep. var."]            = dict(zip(cols, ["Night LST"] * 4 + ["PM2.5"]))
rows["Pixel FE"]             = dict.fromkeys(cols, "Yes")
rows["Country x year FE"]    = dict.fromkeys(cols, "Yes")
rows["Observations"]         = {c: f"{samp(i)[0]:,}" for i, c in enumerate(cols)}
rows["Pixels"]               = {c: f"{samp(i)[1]:,}" for i, c in enumerate(cols)}
rows["Countries (clusters)"] = {c: f"{samp(i)[2]}" for i, c in enumerate(cols)}
rows["Years"]                = {c: f"{samp(i)[3]}-{samp(i)[4]}" for i, c in enumerate(cols)}

tab = pd.DataFrame(rows).T[cols]
tab.index = ["" if i.endswith(" se") else i for i in tab.index]

b1, s1, p1 = get(M["c1pm"], X)
bf, sf, _  = get(M["fspm"], Z)
note = ("SE clustered by country in parentheses. * p<0.1, ** p<0.05, *** p<0.01. "
        "10 km grid, full panel. Cols (4)-(5) restricted to pixel-years with PM2.5.\n"
        f"Col (1) re-estimated on the PM2.5 sample: {b1:.4f}{star(p1)} ({s1:.4f}). "
        f"First stage on the PM2.5 sample: {bf:.4f} ({sf:.4f}), F = {(bf / sf) ** 2:.1f}.")

OUT.mkdir(parents=True, exist_ok=True)
tab.to_csv(OUT / "night_lst_mining_pm25_10km.csv")
(OUT / "night_lst_mining_pm25_10km.md").write_text(tab.to_markdown() + "\n\n" + note + "\n")
(OUT / "night_lst_mining_pm25_10km.tex").write_text(tab.to_latex(escape=True))
print(note)
tab

SE clustered by country in parentheses. * p<0.1, ** p<0.05, *** p<0.01. 10 km grid, full panel. Cols (4)-(5) restricted to pixel-years with PM2.5.
Col (1) re-estimated on the PM2.5 sample: -0.0227** (0.0110). First stage on the PM2.5 sample: 0.0672 (0.0122), F = 30.1.


,(1) TWFE,(2) 2SLS,(3) RF: Temp,(4) TWFE + PM2.5,(5) RF: PM2.5
log(1+NTL),-0.0227**,-0.1040,,-0.0221*,
,(0.0110),(0.1632),,(0.0114),
Mines within 20 km,,,-0.0070,,-0.2710**
,,,(0.0111),,(0.1365)
PM2.5,,,,0.0032*,
,,,,(0.0017),
First stage: Mines 20 km -> log(1+NTL),,0.0672***,,,
,,(0.0122),,,
First-stage F (cluster-robust),,30.1,,,
Dep. var.,Night LST,Night LST,Night LST,Night LST,PM2.5
